In [9]:
import importlib
import helper.common as common

importlib.reload(common)

dspy = common.get_dspy_instance()
llm = common.get_dspy_lm_instance()

## Modules

A DSPy module is a building block for programs that use LMs. DSPy modules are where we apply different prompting frameworks to signatures. We've already been using the basic `Predict` module in our signature examples prior, but there exist many more popular strategies and variants. Here are the current available modules: 

* `dspy.Predict`: Basic predictor. Does not modify the signature. Handles the key forms of learning (i.e., storing the instructions and demonstrations and updates to the LM).

* `dspy.ChainOfThought`: Teaches the LM to think step-by-step before committing to the signature's response. The module automatically adds a "Let's think step by step" prefix to encourage structured thinking. Use this when you need the model to break down complex problems into smaller steps.

* `dspy.ProgramOfThought`: Teaches the LM to output code, whose execution results will dictate the response. Generates executable Python code to solve problems, with built-in error handling and code regeneration capabilities. Use this for mathematical or algorithmic problems that are better solved through actual code execution.

* `dspy.ReAct`: An agent that can use tools to implement the given signature. Implements Reasoning + Acting by interleaving thoughts, actions (via tools), and observations in a structured loop. Use this when your task requires multi-step reasoning and interaction with external tools or APIs.

* `dspy.MultiChainComparison`: Can compare multiple outputs from ChainOfThought to produce a final prediction. Takes multiple reasoning attempts (default 3) and combines them into a single, more accurate response by comparing different reasoning paths. Use this when you need higher accuracy and can afford multiple attempts at solving a problem.

* `majority`: A utility function that takes multiple predictions and returns the most common response after normalizing the text. Use this when you want to implement simple voting among multiple predictions attempts to increase reliability.


---
### ChainOfThought

ChainOfThought works by modifying the prompt signature to include an explicit reasoning step before the output. When initialized with a signature, it creates an extended signature by prepending a "reasoning" field with the prefix "Reasoning: Let's think step by step in order to". This reasoning field forces the language model to write out its thought process before providing the final answer.

In [10]:
from typing import Literal

class Sentiment(dspy.Signature):
    text: str = dspy.InputField()
    sentiment: Literal["happy", "sad", "angry", "confused", "humble"] = dspy.OutputField()


get_sentiment = dspy.ChainOfThought(Sentiment)

In [11]:
response = get_sentiment(text="It's a charming and often affecting journey.")

print("Sentiment: ", response.sentiment)
print("Reason: ", response.reasoning)

Sentiment:  happy
Reason:  The text uses positive descriptors like "charming" and "affecting" in the context of a "journey." "Charming" directly implies pleasantness and delight, which are associated with happiness. "Affecting" in this context likely means emotionally moving or impactful in a positive or poignant way, contributing to a rich and positive experience.


In [12]:
response = get_sentiment(text="I have been playing pubg for 6 hours but my KD is not crossing 1, its feels like to break my mobile")

print("Sentiment: ", response.sentiment)
print("Reason: ", response.reasoning)

Sentiment:  angry
Reason:  The user expresses extreme frustration ("feels like to break my mobile") because their performance (KD) in PUBG is not improving despite playing for a long time (6 hours). This indicates anger.


---
### Program of Thought

ProgramOfThought solves tasks by generating executable Python code rather than working directly with natural language outputs. When given a task, PoT first generates Python code using a ChainOfThought predictor, then executes that code in an isolated Python interpreter. If the code generates any errors, PoT enters a refinement loop where it shows the error to the language model, gets corrected code, and tries executing again, for up to a maximum number of iterations (default 3). The final output comes from actually running the successful code rather than from the language model directly. 

In [13]:
class MathAnalysis(dspy.Signature):
    input_list: list[float] = dspy.InputField(desc="This should be a list of number used for mathamatical calculation")
    required_metrics: list[str] =  dspy.InputField(decs="User specify what metrics he want to calculate like mean, median, mode, average")
    analysis_results: dict[str, float] = dspy.OutputField(decs="Return calculation result with metric and answer")


# Create the module
math_analyzer = dspy.ProgramOfThought(MathAnalysis)

# Example
data = [1.5, 2.8, 3.2, 4.7, 5.1, 2.3, 3.9]
metrics = ['mean', 'median', 'average']

# Run
pot_response = math_analyzer(
    input_list=data,
    required_metrics=metrics
)

In [14]:
print("Reasoning: ", pot_response.reasoning)
print("\nResults: ", pot_response.analysis_results)

Reasoning:  The `code_output` contains a list with a single dictionary, which represents the calculated metrics. I will extract this dictionary directly as the `analysis_results`. The dictionary includes 'mean', 'average', and 'median' as requested in `required_metrics`.

Results:  {'mean': 3.357142857142857, 'average': 3.357142857142857, 'median': 3.2}


In [15]:
llm.inspect_history()





[2025-09-27T19:39:05.958952]

System message:

Your input fields are:
1. `input_list` (list[float]): This should be a list of number used for mathamatical calculation
2. `required_metrics` (list[str]): 
3. `final_generated_code` (str): python code that answers the question
4. `code_output` (str): output of previously-generated python code
Your output fields are:
1. `reasoning` (str): 
2. `analysis_results` (dict[str, float]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## input_list ## ]]
{input_list}

[[ ## required_metrics ## ]]
{required_metrics}

[[ ## final_generated_code ## ]]
{final_generated_code}

[[ ## code_output ## ]]
{code_output}

[[ ## reasoning ## ]]
{reasoning}

[[ ## analysis_results ## ]]
{analysis_results}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "additionalProperties": {"type": "number"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is

---
### Reasoning + Acting (ReAct)

ReAct enables interactive problem-solving by combining reasoning with tool usage. It works by maintaining a trajectory of thought-action pairs, where at each step the model explains its reasoning, selects a tool to use, provides arguments for that tool, and then observes the tool's output to inform its next step. Each iteration consists of four parts: a thought explaining the strategy, selection of a tool name from the available tools, arguments to pass to that tool, and the observation from running the tool. This continues until either the model chooses to "finish" or reaches the maximum number of iterations. Here's a simple example:

In [16]:
# Define a Tool
def wikipedia_search(query: str) -> list[str]:
    """Retrieves abstracts from Wikipedia."""
    # Existing Wikipedia Abstracts Server
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3) 
    return [x['text'] for x in results]

# Define ReAct Module
react_module = dspy.ReAct('question -> response', tools=[wikipedia_search])

# Example
text = "Who won the world series in 1983 and who won the world cup in 1966?"

# Run
react_response = react_module(question=text)

In [17]:
print("Answer: ", react_response.response)
print("\nReasoning: ", react_response.reasoning)

Answer:  The Baltimore Orioles won the World Series in 1983, and England won the World Cup in 1966.

Reasoning:  The user asked for two pieces of information: the winner of the 1983 World Series and the winner of the 1966 World Cup. I used `wikipedia_search` to find both. For the 1983 World Series, the Baltimore Orioles won. For the 1966 World Cup, England won. I have successfully retrieved all the necessary information.


In [18]:
llm.inspect_history()





[2025-09-27T19:41:43.154566]

System message:

Your input fields are:
1. `question` (str): 
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `response`.


User message:

[[ ## question ## ]]
Who won the world series in 1983 and who won the world cup in 1966?

[[ ## trajectory ## ]]
[[ ## thought_0 ## ]]
The user is asking two distinct questions: "Who won the world series in 1983?" and "who won the world cup in 1966?". I need to use `wikipedia_search` twice to get this information. I will start with the World Series.

[[ ## tool_name_0 ## ]]
wikipedia_search

[[ ## tool_args_0 ## ]]


---
### Multi Chain Comparison

MultiChainComparison is a meta-predictor that synthesizes multiple existing completions into a single, more robust prediction. It doesn't generate predictions itself, but instead takes M different completions (default 3) from other predictors - these could be from the same predictor with different temperatures, different predictors entirely, or repeated calls with the same settings. These completions are formatted as "Student Attempt #1:", "Student Attempt #2:", etc., with each attempt packaged as «I'm trying to \[rationale] I'm not sure but my prediction is \[answer]». The module then prompts the model to analyze these attempts holistically with "Accurate Reasoning: Thank you everyone. Let's now holistically..." to synthesize a final answer. This approach helps mitigate individual prediction errors by having the model explicitly compare and critique multiple solution paths before making its final decision.

In [20]:
# Run CoT completions with increasing temperatures
text = "That was phenomenal!"

# Define the Signature and Module
cot_emotion = dspy.ChainOfThought('input -> sentiment: str')

cot_completions = []

for i in range(3):
    # Temperature increases: 0.7, 0.8, 0.9
    temp_config = dict(temperature=0.7 + (0.1 * i))
    completion = cot_emotion(input=text, config=temp_config)
    cot_completions.append(completion)

# Synthesize with MultiChainComparison
mcot_emotion = dspy.MultiChainComparison('input -> sentiment', M=3)
final_result = mcot_emotion(completions=cot_completions, input=text)

print(f"Sentiment: {final_result.sentiment}")
print(f"\nReasoning: {final_result.rationale}")

for i in range(3):
    print(f"\nCompletion {i+1}: ", cot_completions[i])

Sentiment: Positive

Reasoning: The word "phenomenal" is a strong positive adjective, indicating something exceptionally good or impressive. The exclamation mark further amplifies this positive sentiment, leaving no doubt about the speaker's enthusiastic approval.

Completion 1:  Prediction(
    reasoning='The word "phenomenal" expresses extreme positivity and high satisfaction.',
    sentiment='Positive'
)

Completion 2:  Prediction(
    reasoning='The word "phenomenal" strongly conveys a sense of extreme excellence and positivity. This adjective leaves no ambiguity about the speaker\'s very positive feeling.',
    sentiment='Positive'
)

Completion 3:  Prediction(
    reasoning='The word "phenomenal" is a strong positive adjective, and the exclamation mark emphasizes a very positive feeling.',
    sentiment='Positive'
)



---
### Majority

Majority is a utility function that implements a basic voting mechanism across multiple completions to determine the most common answer. It works by taking either a Prediction object (which contains completions) or a list of completions directly, then normalizes their values for the target field (either specified or defaults to the last output field). The normalization process, handled by normalize_text, helps manage slight variations in text that should be considered the same answer (returning None for answers that should be ignored). In cases of ties, earlier completions are prioritized. The function is particularly useful when combined with modules that generate multiple completions (like running predictors with different temperatures) and you want a simple way to find the most common response. The function returns a new Prediction object containing just the winning completion.

In [21]:
# Example Completions From Prior Multi-Chain
majority_result = dspy.majority(cot_completions, field='sentiment')

# Results
print(f"Most common sentiment: {majority_result.sentiment}")

Most common sentiment: Positive
